Konteks / Skenario
Manajemen platform e-commerce meminta dibuatkan dashboard performa cabang toko yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3–4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi join, window function, dan Spark SQL — persis seperti yang dipelajari hari ini.

Menyiapkan Dataset
Jalankan cell berikut untuk membuat dua tabel dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/dimension table).

In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window

# Inisialisasi Spark.
spark = SparkSession.builder.appName("Tugas5_Naufal").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Tabel Target Cabang.
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Generate CSV Lokal.
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/caitlyn/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/caitlyn/tugas5/

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 12:33:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


A. Join & Perbandingan Target (bobot 25%)
Ringkas total pendapat per kota dari df_transaksi, lalu join dengan df_target. Tambahkan kolom Pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

In [2]:
# 1. Baca data dari HDFS.
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/caitlyn/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)

# 2. Tambah kolom pendapatan (unit_terjual x harga_satuan).
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 3. Ringkas total pendapatan per kota lalu join dengan target.
df_ringkasan = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
df_hasil_a = df_ringkasan.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan") * 100)) \
    .orderBy(col("pencapaian_persen").desc())

df_hasil_a.show()

[Stage 5:>                                                        (0 + 12) / 12]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



B. Window Function — Kategori Terlaris per Kota (bobot 25%)
Menggunakan window function, tentukan kategori dengan pendapatan tertinggi di setiap kota (top-1 saja, gunakan row_number()).

In [3]:
# Melakukan agregasi pendapatan per kategori di tiap kota dulu.
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan_kategori"))

# Menggunakan Window Function untuk mengambil top-1 kategori per kota.
window_b = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

df_hasil_b = df_kategori_kota.withColumn("rn", row_number().over(window_b)) \
    .filter(col("rn") == 1) \
    .drop("rn")

df_hasil_b.show()

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



C. Spark SQL (bobot 25%)
Daftarkan df_transaksi dan df_target sebagai temporary view, lalu tulis satu kueri SQL (bukan DataFrame API) yang menampilkan: kota, pic_cabang, dan jumlah transaksi (COUNT) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [4]:
# Mendaftarkan temporary view untuk menjalankan kueri SQL.
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

spark.sql("""
    SELECT t.kota, tar.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tar ON t.kota = tar.kota
    GROUP BY t.kota, tar.pic_cabang
    ORDER BY jumlah_transaksi DESC
""").show()

[Stage 13:==========================================>              (9 + 3) / 12]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



D. Kesimpulan (bobot 25%)
Tulis pada markdown cell (minimal 100 kata): berdasarkan hasil bagian A dan B, cabang mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

Berdasarkan hasil eksekusi query pada Bagian A, cabang dengan kinerja terbaik secara proporsional adalah Purworejo dan Solo, yang secara konsisten mencetak persentase pencapaian (pencapaian_persen) tertinggi dan melampaui batas target yang ditetapkan secara signifikan. Di sisi lain, cabang yang paling memerlukan perhatian manajemen adalah Yogyakarta dan Semarang. Meskipun secara nilai nominal (total_pendapatan) kota-kota besar tersebut mungkin terlihat masif, namun bila dibenturkan dengan target bulanannya (masing-masing 60 juta dan 55 juta), angka persentasenya merupakan yang terendah di antara seluruh cabang. Strategi lokalisasi lebih lanjut seperti mengoptimalkan stok kategori produk terlaris di setiap kota berdasarkan hasil Bagian B, dapat menjadi solusi untuk mendongkrak performa cabang yang tertinggal.